# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides an interactive workflow for loading and exploring the FAIR² mental health survey dataset using the `mlcroissant` library. All entities (record sets, fields, columns, etc.) are referenced by their `@id`.

### Dataset Source
The dataset is described via a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR² Mental Health dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load metadata and dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\n\nDescription: {metadata.description}")
print(f"\nCitation: {getattr(metadata, 'citeAs', 'N/A')}")

## 2. Data Overview
Review available record sets and list their fields using their `@id` for precise referencing.

In [ ]:
# List record sets and their fields/columns using @id

record_sets = list(dataset.record_sets())  # Each is a RecordSetMetadata object
print("Record sets present in the dataset:")
for rs in record_sets:
    print(f"  Record set name: {rs.name} | @id: {rs.id}")
    for field in rs.fields:
        print(f"    Field: {field.name} | @id: {field.id} | dataType: {field.data_type}")

## 3. Data Extraction
Load the records from each record set into a pandas DataFrame. Use record set and field `@id`s from the above overview.

For this dataset, the main survey record set is typically named something like `kilifi_survey_data` (verify the actual name and `@id` in the output above).

In [ ]:
# Extract records from each record set by @id and load into dataframes for analysis

# Build a dictionary of DataFrames keyed by record set @id
dataframes = {}

for rs in record_sets:
    rs_id = rs.id  # e.g., 'cr:KilifiSurveyRecords'
    print(f"Loading record set: {rs.name} | @id: {rs_id}")
    records = list(dataset.records(record_set=rs_id))  # yields dicts with field @id keys
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Shape: {df.shape}\n")

# Pick main survey data record set (replace with the actual @id printed above)
main_rs_id = list(dataframes.keys())[0]  # Use the actual main survey record set @id
print(f"Using main survey record set: {main_rs_id}")
print(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing: filter records, normalize a numeric field, group by a demographic field, etc. All field references are by their `@id`.

*Example fields:*
- Numeric: `gad7_total_score` (Generalized Anxiety), `phq9_total_score` (Depression), `psq_total_score` (Perceived Stress).  
- Demographics: `sex_at_birth`, `age`, `village`, `level_of_education`.

In [ ]:
# Example: Analyze GAD-7 total scores by sex at birth

# Use @id for fields
gad7_score_field = 'gad7_total_score'    # Replace by the correct @id found in overview if differs
sex_field = 'sex_at_birth'               # Replace by @id found in field overview

df = dataframes[main_rs_id]

# Drop missing GAD-7 scores
filtered_df = df[df[gad7_score_field].notna()].copy()
filtered_df[gad7_score_field] = pd.to_numeric(filtered_df[gad7_score_field], errors='coerce')

# Remove outliers (e.g., only valid 0-21 for GAD-7)
filtered_df = filtered_df[(filtered_df[gad7_score_field] >= 0) & (filtered_df[gad7_score_field] <= 21)]

# Normalize GAD-7 scores
filtered_df["gad7_normalized"] = (
    filtered_df[gad7_score_field] - filtered_df[gad7_score_field].mean()
) / filtered_df[gad7_score_field].std()

print(f"Normalized GAD-7 scores (first 5):\n", filtered_df[[gad7_score_field, "gad7_normalized"]].head())

# Group by sex
if sex_field in filtered_df.columns:
    group_df = filtered_df.groupby(sex_field)[gad7_score_field].agg(['mean', 'std', 'count'])
    print(f"\nMean GAD-7 scores by {sex_field}:")
    print(group_df)

## 5. Visualization
Visualize the distribution of GAD-7 scores by sex at birth for the Kilifi survey respondents.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of GAD-7 scores, grouped by sex
if sex_field in filtered_df.columns:
    plt.figure(figsize=(8, 5))
    sns.boxplot(x=sex_field, y=gad7_score_field, data=filtered_df, palette="pastel")
    plt.title("GAD-7 Score Distribution by Sex at Birth")
    plt.xlabel("Sex at Birth (@id: {}".format(sex_field))
    plt.ylabel("GAD-7 Total Score (@id: {}".format(gad7_score_field))
    plt.show()
else:
    print(f"Field {sex_field} not found in DataFrame columns.")

## 6. Conclusion
In this notebook, we've:
- Loaded the FAIR² Kilifi Mental Health Survey dataset via its Croissant metadata using `mlcroissant`.
- Inspected all record sets, fields, and referenced all elements by their `@id`.
- Carried out exploratory analysis and simple EDA of GAD-7 anxiety scores by demographic group.
- Visualized score distributions.

__This approach can be extended to the PHQ-9 or PSQ and other fields by swapping field `@id`s as needed, as listed in section 2 above.__

For further analysis, explore grouping by other fields (e.g., `level_of_education`, `village`) or creating additional visualizations to surface patterns in mental health responses across the surveyed population.